In [1]:
from urllib.request import urlopen

import numpy as np
import torch
from torch import nn

torch.set_num_threads(2)
torch.manual_seed(7)

url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/AirPassengers.csv"
with urlopen(url, timeout=60) as response:
    passengers = torch.tensor(np.loadtxt(response, delimiter=",", skiprows=1, usecols=2)).float()

split = 120
mean, scale = passengers[:split].mean(), passengers[:split].std()
windows = ((passengers - mean) / scale).unfold(0, 13, 1)
x, y = windows[:, :12].unsqueeze(-1), windows[:, 12]
train_size = split - 12

rnn = nn.RNN(1, 24, batch_first=True, bidirectional=True)
head = nn.Linear(48, 1)
optimizer = torch.optim.Adam([*rnn.parameters(), *head.parameters()], lr=0.01)

def forecast(history):
    _, state = rnn(history)
    return head(torch.cat((state[0], state[1]), dim=1)).squeeze(1)

for epoch in range(350):
    optimizer.zero_grad()
    loss = nn.functional.mse_loss(forecast(x[:train_size]), y[:train_size])
    loss.backward()
    optimizer.step()

with torch.no_grad():
    predictions = forecast(x[train_size:]) * scale + mean
actual = passengers[split:]
print(f"Rolling one-month BiRNN MAE (thousands): {(predictions - actual).abs().mean():.1f}")
print(f"Same month last year MAE (thousands): {(passengers[split-12:-12] - actual).abs().mean():.1f}")
print("Actual:  ", actual[:6].int().tolist())
print("Forecast:", predictions[:6].round().int().tolist())

Rolling one-month BiRNN MAE (thousands): 53.1
Same month last year MAE (thousands): 47.6
Actual:   [360, 342, 406, 396, 420, 472]
Forecast: [343, 324, 374, 362, 382, 456]
